# Symbolic spin operators with SpynDynamica

SpynDynamica is a Python port in progress of [SpinDynamica](https://www.spindynamica.soton.ac.uk), Malcolm
Levitt's Mathematica package. It is a separate import (`import spyndynamica`) that uses NumPy and SymPy and
nothing of Spynach. This guide covers the symbolic core and Liouville space: operators are sums of product
operators and spherical tensor operators with symbolic coefficients, and products, commutators, rotations and
weak-coupling evolution are exact algebra; operator bases and superoperators come on top, and axis systems,
Euler angles and quaternions describe frames. The numeric engine (pulse sequences with strong coupling and
relaxation, signals and spectra, time-dependent Hamiltonians, powder averages, chemical exchange) is in
[Simulations with SpynDynamica](spyndynamica-simulations.ipynb); operator algebras, permutation symmetry and
three-dimensional pictures are in [Algebras, symmetry and pictures](spyndynamica-symmetry.ipynb). The table of
SpinDynamica names and the list of differences are in [Coming from SpinDynamica](coming-from-spindynamica.ipynb).

SpinDynamica is the work of Malcolm H. Levitt and co-authors. Its [terms of use](https://www.spindynamica.soton.ac.uk/about/termsofuse/)
state that it is open-source and freely available for use, and ask that published work cite
C. Bengs, M. H. Levitt, *SpinDynamica: Symbolic and numerical magnetic resonance in a Mathematica environment*, Magn. Reson. Chem. (2017), doi:10.1002/mrc.4642. Work that uses SpynDynamica should cite that publication.

Every convention below is the one of SpinDynamica 3.10.2 and is checked against it
(`tests/spyndynamica/`, 887 reference cases generated from the Mathematica package).

*Generated from [`docs/spyndynamica.md`](../spyndynamica.md) by `docs/build_notebooks.py`; edit the Markdown guide, not this notebook.*

Run the cells in order: later cells use names defined by earlier ones. The saved outputs come from running the notebook top to bottom; plotting cells need Plotly (`pip install -e ".[docs]"`).

## Contents

1. [Spin systems and operators](#Spin-systems-and-operators)
2. [Rotations and pulses](#Rotations-and-pulses)
3. [A product-operator calculation](#A-product-operator-calculation)
4. [Matrices and numbers](#Matrices-and-numbers)
5. [Bases, kets and bras](#Bases,-kets-and-bras)
6. [Spins above 1/2 and tensor operators](#Spins-above-1/2-and-tensor-operators)
7. [Superoperators and operator bases](#Superoperators-and-operator-bases)
8. [Axis systems, Euler angles and quaternions](#Axis-systems,-Euler-angles-and-quaternions)

## Spin systems and operators

A spin system is an object, and several can exist side by side. `SpinSystem(2)` is two spins-1/2 with the
labels 1 and 2. Operators come from the system, `@` is the operator product and `*` multiplies by a scalar.

In [1]:
import sympy as sp

from spyndynamica import SpinSystem, commutator

s = SpinSystem(2)
I = s.I

print(commutator(I(1, "x"), I(1, "y")))
print(I(1, "x") @ I(2, "z") + 3 * I(1, "y"))
print(I(1, "+"), "|", I(None, "z"), "|", s.dot(1, 2))

I*I1z
3*I1y + I1x I2z
I1x + I*I1y | I2z + I1z | I1x I2x + I1y I2y + I1z I2z


`I(spin, component)` takes `'x'`, `'y'`, `'z'`, `'+'`, `'-'`, `'alpha'` and `'beta'`, an angle (the component in
the xy plane at that phase) or polar angles `(θ, φ)`. A list of spins, or `None` for all of them, gives the sum.
`s.dot(1, 2)` is the scalar product `I1·I2`. In a notebook an operator is typeset.

An operator knows its trace, norm, coherence orders, and how much of another operator it contains:

In [2]:
rho = 2 * I(1, "x") @ I(2, "z") + 3 * I(1, "y")
print(rho.amplitude(I(1, "x") @ I(2, "z")), rho.amplitude(I(1, "y")))
print((I(1, "+") @ I(2, "+")).coherence_orders(), I(1, "x").coherence_orders())
print(I(1, "x").norm(), s.unity().trace())

2 3
[2] [-1, 1]
1 4


`a.amplitude(b)` is `(b|a)/(b|b)` with the Liouville bracket `(a|b) = Tr(a† b)`, SpinDynamica's
`OperatorAmplitude[a -> b]`.

## Rotations and pulses

`rotate(angle, phase)` returns `R A R†` for `R = exp(−i angle n·I)`. The axis lies in the xy plane at `phase`
(`'x'`, `'y'`, `'-x'`, `'-y'` or an angle), is `'z'`, or has the polar angles `axis=(θ, φ)`. `spins` restricts
the rotation; the default is all spins.

In [3]:
beta, phi = sp.symbols("beta phi", real=True)

print(I(1, "z").rotate(sp.pi / 2, "x"))
print(I(1, "z").rotate(beta, phi, spins=[1]))
print((I(1, "z") @ I(2, "z")).rotate(sp.pi / 2, "y", spins=[1]))
print(I(1, "x").rotate_euler(sp.pi / 2, sp.pi / 2, 0))

-I1y
sin(beta)*sin(phi)*I1x - sin(beta)*cos(phi)*I1y + cos(beta)*I1z
I1x I2z
-I1z


`rotate_euler(α, β, γ)` is the zyz rotation `Rz(α) Ry(β) Rz(γ)`. The rotation operator itself is
`s.rotation(angle, phase)`:

In [4]:
print(s.rotation(beta, "x", spins=[1]))

cos(beta/2) - 2*I*sin(beta/2)*I1x


## A product-operator calculation

`evolve(H, t)` is `exp(−iHt) ρ exp(+iHt)` in closed form when the terms of `H` commute, which covers chemical
shifts and weak couplings of spins-1/2. A 90° pulse on spin 1, then evolution:

In [5]:
omega, J, t, tau = sp.symbols("Omega1 J t tau", real=True)
H = omega * I(1, "z") + 2 * sp.pi * J * (I(1, "z") @ I(2, "z"))

rho = I(1, "z").rotate(sp.pi / 2, "x", spins=[1]).evolve(H, t)
print(rho)
print(rho.amplitude(2 * I(1, "x") @ I(2, "z")))

sin(Omega1*t)*cos(pi*J*t)*I1x - cos(Omega1*t)*cos(pi*J*t)*I1y + 2*sin(pi*J*t)*cos(Omega1*t)*I1x I2z + 2*sin(Omega1*t)*sin(pi*J*t)*I1y I2z
sin(pi*J*t)*cos(Omega1*t)


A spin echo refocuses the shift and keeps the coupling, and INEPT turns the antiphase term into magnetisation
of the second spin:

In [6]:
start = I(1, "z").rotate(sp.pi / 2, "x", spins=[1])
echo = start.evolve(H, tau).rotate(sp.pi, "y").evolve(H, tau).simplify()
print(echo)

inept = echo.rotate(sp.pi / 2, "y", spins=[1]).rotate(sp.pi / 2, "x", spins=[2]).simplify()
print(inept.amplitude(2 * I(1, "z") @ I(2, "y")))

-cos(2*pi*J*tau)*I1y + 2*sin(2*pi*J*tau)*I1x I2z
sin(2*pi*J*tau)


A Hamiltonian whose terms do not commute (strong coupling) raises `NotImplementedError`: that case belongs to
the numeric engine ([Simulations with SpynDynamica](spyndynamica-simulations.ipynb)).

## Matrices and numbers

`matrix()` is the exact SymPy matrix and `numeric(values)` a NumPy array with numbers put in for the symbols.
In the Zeeman basis every spin runs from `m = +I` down and the first spin changes fastest, so two spins-1/2
are ordered αα, βα, αβ, ββ.

In [7]:
print(I(1, "x").matrix())
print(H.numeric({omega: 2 * sp.pi * 100, J: 7}).real.diagonal().round(2))
print(s.from_matrix(I(1, "x").matrix() * I(2, "z").matrix()))

Matrix([[0, 1/2, 0, 0], [1/2, 0, 0, 0], [0, 0, 0, 1/2], [0, 0, 1/2, 0]])
[ 325.15 -325.15  303.16 -303.16]
I1x I2z


`s.from_matrix` goes back from a matrix to the operator.

## Bases, kets and bras

`s.zeeman` and `s.singlet_triplet()` are bases; the second is for two spins-1/2, in SpinDynamica's order
`S0, T+1, T0, T-1`. A matrix can be taken in either, and states are numbered from 1 or named.

In [8]:
st = s.singlet_triplet()
print(st.ket("S0"))
print(s.dot(1, 2).matrix(st))
print(st.transformation(s.zeeman))

up, down = sp.Rational(1, 2), -sp.Rational(1, 2)
a = s.ket(up, up)
print(I(1, "-") @ a, "|", a.dagger() @ a, "|", a @ s.ket(down, up).dagger())

(-sqrt(2)/2)*|-1/2, 1/2> + (sqrt(2)/2)*|1/2, -1/2>
Matrix([[-3/4, 0, 0, 0], [0, 1/4, 0, 0], [0, 0, 1/4, 0], [0, 0, 0, 1/4]])
Matrix([[0, -sqrt(2)/2, sqrt(2)/2, 0], [1, 0, 0, 0], [0, sqrt(2)/2, sqrt(2)/2, 0], [0, 0, 0, 1]])
|-1/2, 1/2> | 1 | 1/2*I1x + I/2*I1y + I1x I2z + I*I1y I2z


Three more bases come from the system or from another basis: `s.angular_momentum_basis()` couples the spins to
eigenkets of the total angular momentum, `s.eigenbasis(H)` holds the eigenkets of an operator, and
`basis.transformed(op)` applies an operator to the kets of a basis.

In [9]:
coupled = s.angular_momentum_basis()
print(coupled.quantum_numbers)
print(s.dot(1, 2).matrix(coupled).diagonal())
print(st.ket("T0").magnetic_quantum_numbers(), I(1, "x").normalized().norm())

[(1, 1, (1/2, 1)), (1, 0, (1/2, 1)), (1, -1, (1/2, 1)), (0, 0, (1/2, 0))]
Matrix([[1/4, 1/4, 1/4, -3/4]])
[0] 1


A basis of your own comes from `define_basis(kets, labels)`, which requires the kets to be orthonormal
(`check_basis(kets)` says what a set of kets is). `product_basis` combines bases of separate sets of spins: the
kets of the first basis change fastest, and `system` is the spin system of the result.

In [10]:
from spyndynamica import define_basis, product_basis

k = s.ket
triplets_first = [k(up, up), (k(up, down) + k(down, up)) / sp.sqrt(2), k(down, down), (k(up, down) - k(down, up)) / sp.sqrt(2)]
mine = define_basis(triplets_first, labels=["T+", "T0", "T-", "S"])
print(s.dot(1, 2).matrix(mine).diagonal(), "|", mine.ket("S"))

three = SpinSystem(3)
first_two, third = SpinSystem([(1, up), (2, up)]), SpinSystem([(3, up)])
st_and_zeeman = product_basis(first_two.singlet_triplet(), third.zeeman, system=three)
print(three.dot(1, 2).matrix(st_and_zeeman).diagonal())

Matrix([[1/4, 1/4, 1/4, -3/4]]) | (-sqrt(2)/2)*|-1/2, 1/2> + (sqrt(2)/2)*|1/2, -1/2>
Matrix([[-3/4, 1/4, 1/4, 1/4, -3/4, 1/4, 1/4, 1/4]])


`ket @ bra` is an operator, `bra @ ket` a number and `operator @ ket` a ket. The operators of basis states are
`s.population(r)`, `s.coherence(r, s)` and `s.single_transition(r, s, 'x')`; `s.permutation([1, 2])` exchanges
spins.

## Spins above 1/2 and tensor operators

Any spin quantum numbers can be mixed. A spin above 1/2 is written in spherical tensor operators `T(k, q)`,
which is also how its operators print (`T1(2,0)` is rank 2, order 0, of spin 1).

In [11]:
d = SpinSystem([1])
print(d.I(1, "z") @ d.I(1, "z"))
print(d.T(1, 2, 0).rotate(beta, "y"))

pair = SpinSystem({"H": 0.5, "D": 1})
print(pair.dimension, pair.I("H", "z") @ pair.I("D", "+"))
print(s.T([1, 2], 2, 0), "|", s.T([1, 2], 0, 0))

2/3 + sqrt(6)/3*T1(2,0)
sqrt(6)*sin(2*beta)/4*T1(2,-1) + sqrt(6)*sin(beta)**2/4*T1(2,-2) + (3*cos(2*beta)/4 + 1/4)*T1(2,0) - sqrt(6)*sin(2*beta)/4*T1(2,1) + sqrt(6)*sin(beta)**2/4*T1(2,2)
6 -sqrt(2)*IHz TD(1,1)
-sqrt(6)/6*I1x I2x - sqrt(6)/6*I1y I2y + sqrt(6)/3*I1z I2z | -sqrt(3)/3*I1x I2x - sqrt(3)/3*I1y I2y - sqrt(3)/3*I1z I2z


`s.T(spin, k, q)` is SpinDynamica's `opT`: `T(1, ±1) = ∓I±/√2`, `T(2, 0) = (3Iz² − I(I+1))/√6`. With a pair of
spins it couples their rank-1 operators, so `T(0, 0) = −I1·I2/√3`.

More spins, or single-spin operators of other ranks (`ranks`), are coupled one after the other from left to
right with Clebsch-Gordan coefficients. When several sequences of intermediate ranks lead to the same total
rank, `history` names the rank after each coupling step; `rank_histories` lists the sequences.

In [12]:
from spyndynamica import rank_histories

print(rank_histories([1, 1, 1]))
print(three.T([1, 2, 3], 3, 3), "|", three.T([1, 2, 3], 0, 0, history=(1, 0)))
print(pair.T(["H", "D"], 3, 3, ranks=(1, 2)))

[(0, 1), (1, 0), (1, 1), (1, 2), (2, 1), (2, 2), (2, 3)]
-sqrt(2)/4*I1x I2x I3x - sqrt(2)*I/4*I1x I2x I3y - sqrt(2)*I/4*I1x I2y I3x + sqrt(2)/4*I1x I2y I3y - sqrt(2)*I/4*I1y I2x I3x + sqrt(2)/4*I1y I2x I3y + sqrt(2)/4*I1y I2y I3x + sqrt(2)*I/4*I1y I2y I3y | -sqrt(6)*I/6*I1x I2y I3z + sqrt(6)*I/6*I1x I2z I3y + sqrt(6)*I/6*I1y I2x I3z - sqrt(6)*I/6*I1y I2z I3x - sqrt(6)*I/6*I1z I2x I3y + sqrt(6)*I/6*I1z I2y I3x
-sqrt(2)/2*IHx TD(2,2) - sqrt(2)*I/2*IHy TD(2,2)


The Wigner matrices and Clebsch-Gordan coefficients are available in the same conventions:

In [13]:
from spyndynamica import clebsch_gordan, wigner_d, wigner_D

print(wigner_d(1, beta))
print(clebsch_gordan(1, 1, 1, -1, 2, 0), wigner_D(1, 0, sp.pi / 2, 0)[0, 2])

Matrix([[cos(beta)/2 + 1/2, -sqrt(2)*sin(beta)/2, 1/2 - cos(beta)/2], [sqrt(2)*sin(beta)/2, cos(beta), -sqrt(2)*sin(beta)/2], [1/2 - cos(beta)/2, sqrt(2)*sin(beta)/2, cos(beta)/2 + 1/2]])
sqrt(6)/6 1/2


## Superoperators and operator bases

A superoperator maps operators to operators. It is built from operators, acts exactly, and composes with `@`.

In [14]:
from spyndynamica import commutation, default_operator_basis, lindbladian, phenomenological_relaxation

print(commutation(I(1, "z"))(I(1, "x")))
print(lindbladian(I(1, "-"))(s.unity()))

T1, T2 = sp.symbols("T1 T2", positive=True)
relaxation = phenomenological_relaxation(s, T1, T2)
print(relaxation(I(1, "z")), "|", relaxation(I(1, "+") @ I(2, "z")))

I*I1y
-2*I1z
-1/T1*I1z | (-1/T2 - 1/T1)*I1x I2z + (-I/T2 - I/T1)*I1y I2z


Besides `commutation` there are `left_multiplication`, `right_multiplication`, `double_commutation`,
`projection`, `coherence_filter`, `rotation_superoperator` and `permutation_superoperator`, and
`liouvillian(H, relaxation)` is `−i[H, ·]` plus the relaxation. `phenomenological_relaxation` is the
uncorrelated model for spins-1/2: every `Iz` in a product adds `1/T1` and every `I±` adds `1/T2`.

A superoperator has a matrix in an operator basis, `S_ij = (B_i | S B_j)`. The default basis is SpinDynamica's
`ShiftAndZOperatorBasis` sorted by coherence order.

In [15]:
basis = default_operator_basis(s)
print([str(b) for b in basis.operators[:4]])
print(commutation(I(1, "z")).numeric(basis=basis).real.diagonal())
print(basis.vector(I(1, "x")).T)

['I1x I2x - I*I1x I2y - I*I1y I2x - I1y I2y', 'sqrt(2)*I1x I2z - sqrt(2)*I*I1y I2z', 'sqrt(2)*I1z I2x - sqrt(2)*I*I1z I2y', 'sqrt(2)/2*I1x - sqrt(2)*I/2*I1y']
[-1. -1.  0. -1.  0. -1.  1.  0.  0.  0.  0.  1.  0.  1.  0.  1.]
Matrix([[0, 0, 0, sqrt(2)/2, 0, 0, 0, 0, 0, 0, 0, 0, 0, sqrt(2)/2, 0, 0]])


The other bases are `cartesian_product_basis`, `shift_and_polarization_basis`, `zeeman_ket_bra_basis`,
`singlet_triplet_operator_basis` and `spherical_tensor_basis`.

`spherical_tensor_basis` holds the normalised irreducible tensor operators of any spin system, coupled as
`s.T` couples them; `basis.indices` lists their `(rank, order)`. `spherical_component_filter(s, rank, to,
source)` is built on them. It takes the component `source` of every tensor of that rank to its component `to`
and removes everything else; with one component it is the projector onto that rank and order.

In [16]:
from spyndynamica import spherical_component_filter, spherical_tensor_basis

print(spherical_tensor_basis(s).indices)
mixture = I(1, "z") + 2 * I(1, "z") @ I(2, "z") + I(1, "x") @ I(2, "y")
print(spherical_component_filter(s, 1, 0)(mixture))
print(spherical_component_filter(s, 2, 0)(mixture), "|", spherical_component_filter(s, 1, 1, 0)(I(1, "z")))

[(2, -2), (1, -1), (1, -1), (1, -1), (2, -1), (0, 0), (0, 0), (1, 0), (1, 0), (1, 0), (2, 0), (1, 1), (1, 1), (1, 1), (2, 1), (2, 2)]
I1z + 1/2*I1x I2y - 1/2*I1y I2x
-2/3*I1x I2x - 2/3*I1y I2y + 4/3*I1z I2z | -sqrt(2)/2*I1x - sqrt(2)*I/2*I1y


An operator basis of your own is `define_operator_basis(operators)`, which requires the operators to be
orthonormal (`check_operator_basis` says what a set is). An existing basis is reordered with
`basis.reordered(order)`, where positions count from 0, and transformed with `basis.transformed(superoperator)`,
for which the superoperator must be unitary:

In [17]:
from spyndynamica import cartesian_product_basis, rotation_superoperator

cartesian = cartesian_product_basis(s)
turned = cartesian.transformed(rotation_superoperator(s, sp.pi / 2, "y"))
print([str(b) for b in turned.operators[1:4]], turned.check())
print(turned.vector(I(1, "x")).T)

['-I1z', 'I1y', 'I1x'] orthonormal
Matrix([[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])


## Axis systems, Euler angles and quaternions

These routines work with numbers. `axis_system` builds a right-handed orthonormal set of axes, as the rows of
an array, from a z axis, from an x and a z axis (the x axis of the result lies in their plane) or from all
three. `axes_to_euler(system2, system1)` gives the Euler angles of the rotation `Rz(α) Ry(β) Rz(γ)` that
brings every axis of the first argument onto the corresponding axis of the second (default: the laboratory
axes). With the principal axis system of an interaction and a reference frame these are the angles that go into
the spherical tensor form of a Hamiltonian.

In [18]:
import numpy as np

from spyndynamica import axes_to_euler, axis_system, rotate_euler

principal = axis_system([1, 2, 0], [1, 1, 1])
molecular = axis_system([0, 1, 1], [-1, 2, 0.5])
omega = axes_to_euler(principal, molecular)
print(np.round(omega, 4))
print(np.allclose(rotate_euler(principal, omega, principal), molecular))

[1.0472 1.1832 2.6706]
True


`rotate_euler(vectors, angles)` rotates about the laboratory axes; with a third argument it rotates about the
axes of that system, by `γ` about its z axis, then by `β` about its y axis, then by `α` about its z axis.
`rotate_polar(vectors, ξ, direction)` rotates through `ξ` about an axis given by its polar angles `(θ, φ)` or
as a vector, and `polar_rotation_matrix(ξ, θ, φ)` is the matrix, which may be symbolic. A rotation is also a
unit quaternion `(cos ξ/2, n sin ξ/2)`:

In [19]:
from spyndynamica import polar_rotation_matrix, polar_to_quaternion, quaternion_product, quaternion_to_euler
from spyndynamica import quaternion_to_polar, rotate_polar

print(rotate_polar([1, 0, 0], np.pi / 2, [0, 0, 1]).round(3))
print(polar_rotation_matrix(sp.Symbol("xi"), 0, 0))
quarter_z = polar_to_quaternion(np.pi / 2, 0, 0)
quarter_x = polar_to_quaternion(np.pi / 2, np.pi / 2, 0)
both = quaternion_product(quarter_x, quarter_z)
print(both.round(4), np.round(quaternion_to_polar(both), 4), np.round(quaternion_to_euler(both), 4))

[0. 1. 0.]
Matrix([[cos(xi), -sin(xi), 0], [sin(xi), cos(xi), 0], [0, 0, 1]])
[ 0.5  0.5 -0.5  0.5] [ 2.0944  0.9553 -0.7854] [-1.5708  1.5708 -3.1416]


`quaternion_product(q2, q1)` belongs to the rotation `q1` followed by `q2`. `vector_to_polar` and
`polar_to_vector` convert between a vector and its length and polar angles. For molecular geometry there are
`distance`, `bond_angle`, `subtended_angle`, `torsion_angle` and `solid_angle`:

In [20]:
from spyndynamica import bond_angle, distance, solid_angle, torsion_angle

carbon, hydrogens = [0, 0, 0], [[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]]
print(round(distance(carbon, hydrogens[0]), 4), round(np.degrees(bond_angle(hydrogens[0], carbon, hydrogens[1])), 2))
print(round(np.degrees(torsion_angle([1, 0, 0], [0, 0, 0], [0, 1, 0], [0, 1, 1])), 1))
print(round(solid_angle(hydrogens[:3]) / np.pi, 4), "π")

1.7321 109.47
-90.0
1.0 π
